# Chapter 14a — GPU Matmul Deep Dive

> Course: **llm.c — Zero to Hero**, companion to Chapter 14.
> Builds on Chapter 4 (CPU matmul), Chapter 11 (coalescing & vectorized loads), and Chapter 14 (calling cuBLAS).

Chapter 14 gave you one rule: **don't write your own matmul — call cuBLAS.** That rule is correct, but it leaves a hole. If cuBLAS is just a function call, *why* did it take NVIDIA two decades and tens of thousands of lines to write? And what is it actually doing while your program waits?

This companion fills that hole. We are not going to beat cuBLAS. We *are* going to build a clear mental picture of what a fast GPU matmul looks like inside, so that the rest of the course — fused kernels, mixed precision, attention — stops feeling like magic.

Matmul is worth this much attention because it is, by a wide margin, the most important operation in deep learning. On a typical large language model, **more than 80% of the running time is spent multiplying matrices**. A 10% faster matmul is roughly an 8% faster model — which, at data-center scale, is millions of dollars. Everything else in `llm.c` exists to feed these multiplications.

### What you'll learn

- What a GEMM (general matrix-multiply) really computes, in one picture.
- Why a *naive* GPU matmul is slow even though the GPU is fast — the **memory wall** and **arithmetic intensity**.
- The ladder of optimizations real kernels climb: coalescing → shared-memory tiling → register tiling → vectorized loads → warp tiling → tensor cores.
- What **tensor cores** do and why mixed precision exists.
- How `cublasLt` **fuses** bias and activation into the matmul (the trick `llm.c` leans on).
- How **batched / strided-batched** GEMM powers attention.

You'll finish by benchmarking a naive kernel against cuBLAS on this machine and *seeing* the gap.


## 1. The one operation that matters

Let's anchor the stakes before any code.

A Transformer is, computationally, a tall stack of matrix multiplies with small bits of glue (add, normalize, activate) in between. When people profile a model like Llama-8B, they find that **over 80% of the wall-clock time** is inside some flavor of matrix multiply. The attention scores are a matmul. The projections are matmuls. The feed-forward network is two big matmuls. Even the final vocabulary projection is a matmul.

So the single most valuable thing you can do for training or inference speed is make matmul fast. That is exactly why NVIDIA ships a hand-tuned library (cuBLAS) and why `llm.c` routes every dense linear layer through it.

The flip side: matmul is also the operation with the most *room* to be slow. A first attempt can easily run at **a small fraction of the hardware's peak**. Closing that gap is what the rest of this chapter is about.


## 2. What a GEMM actually computes

GEMM stands for **GE**neral **M**atrix **M**ultiply: `C = A @ B` (optionally scaled and with a `C` added, but ignore that for now).

- `A` has shape `(M, K)`.
- `B` has shape `(K, N)`.
- `C` has shape `(M, N)`.

The rule for a single output cell is the dot product of one row of `A` with one column of `B`:

$$C[i, j] = \sum_{k=0}^{K-1} A[i, k] \cdot B[k, j]$$

That's it. The whole result is just `M × N` of these dot products, each `K` long.

![GEMM as inner products](course/figures/fig_14a_gemm_inner_products.png)

The CPU version is four lines you've already seen (it's `matmul_forward_cpu` in [`dev/cuda/matmul_forward.cu`](dev/cuda/matmul_forward.cu)):

```c
for (int i = 0; i < M; i++)
  for (int j = 0; j < N; j++) {
    float acc = 0.0f;
    for (int k = 0; k < K; k++) acc += A[i*K + k] * B[k*N + j];
    C[i*N + j] = acc;
  }
```

**The amount of arithmetic is fixed:** `2 * M * N * K` floating-point operations (one multiply and one add per term). No clever kernel does fewer multiplies. So if everyone does the same arithmetic, why are some kernels 100× faster than others? The answer is *not* about compute — it's about **memory**.


## 3. Why the naive GPU kernel is slow

The obvious GPU kernel gives one thread to each output cell. Each thread reads a full row of `A` and a full column of `B` from global memory, multiplies them, writes one number. That is `matmul_forward_kernel1` in [`dev/cuda/matmul_forward.cu`](dev/cuda/matmul_forward.cu):

```c
int row = blockIdx.x * blockDim.x + threadIdx.x;   // which output row
int col = blockIdx.y * blockDim.y + threadIdx.y;   // which output col
float acc = 0.0f;
for (int k = 0; k < K; k++) acc += A[row*K + k] * B[k*N + col];
C[row*N + col] = acc;
```

It is correct. It is also slow, and the reason is a number called **arithmetic intensity**: how many math operations you do per byte you fetch from slow memory.

Each inner-loop step does **2 FLOPs** (one multiply, one add) but reads **8 bytes** (two `float`s). That's an intensity of `2 / 8 = 0.25` FLOPs per byte. The GPU can do tens of TFLOPs of math but can only *read* memory at hundreds of GB/s. At 0.25 FLOP/byte the math units sit idle, starving for data. The kernel is **memory-bound**.

Worse, the naive kernel re-reads the same data constantly. Every thread in a row re-reads the same row of `A`; every thread in a column re-reads the same column of `B`. The same bytes cross the memory bus hundreds of times.


### The memory hierarchy is the whole game

A GPU doesn't have one kind of memory — it has a pyramid, fast-and-tiny at the top, slow-and-huge at the bottom:

![GPU memory hierarchy](course/figures/fig_14a_memory_hierarchy.png)

The numbers that matter: registers and shared memory are **10–50× faster** than global memory (HBM/GDDR). The naive kernel only ever touches the slow bottom layer. Every good matmul kernel is, at heart, a scheme to **pull each piece of data up the pyramid once and reuse it many times** before letting it go.

You can see the same story on a *roofline* plot. The diagonal line is the most throughput the memory system can sustain at a given arithmetic intensity; the flat ceiling is the chip's peak compute. Low intensity pins you to the diagonal (memory-bound); high intensity lets you reach the ceiling (compute-bound).

![Roofline: naive vs cuBLAS](course/figures/fig_14a_roofline.png)

The naive kernel sits far down the left ramp. cuBLAS, by reusing data aggressively, pushes its intensity high enough to bump against the compute ceiling. **The entire optimization journey is moving that dot to the right.**


## 4. The optimization ladder

Here is the path real kernels (and cuBLAS internally) climb to go from "a few percent of peak" to "near peak." You won't write these — but knowing the names and the one-line idea behind each is what turns cuBLAS from a black box into a glass box. The rough numbers below are from public SGEMM walkthroughs on a modern GPU; treat them as relative, not absolute.

| Rung | Idea in one sentence | Relative speed |
|---|---|---|
| **1. Naive** | One thread per output cell, everything from global memory. | 1× (baseline) |
| **2. Coalescing** | Renumber threads so neighbors read *neighboring* addresses, turning many small memory transactions into a few wide ones. | ~7× |
| **3. Shared-memory tiling** | A block cooperatively loads a tile of `A` and `B` into shared memory, then every thread reads from there instead of global memory. | ~8× |
| **4. 1D register tiling** | Each thread computes *several* output cells, holding reused values in registers. | ~30× |
| **5. 2D register tiling** | Each thread computes a small `TM×TN` block as an outer product — every value loaded from shared memory gets used `TM×TN` times. | ~60× |
| **6. Vectorized loads** | Use 128-bit `float4` loads/stores so each instruction moves 16 bytes (Chapter 11's trick). | ~70× |
| **7. Warp tiling** | Match the tile layout to how warps physically execute, cutting shared-memory bank conflicts. | ~80× |
| **8. Tensor cores** | Hand whole sub-matrix multiplies to dedicated hardware units — one instruction does a 16×16×16 multiply. | 30×+ again |

Notice the shape of the story: rungs 2–7 are all about **memory** — coalesce it, cache it in shared memory, cache it again in registers, move it in wide chunks. Only at the very top (tensor cores) do we change the *compute*. That's the lesson of GPU performance in one table: **feed the math units; the math is rarely the bottleneck.**


### Tiling, the key idea, in one picture

Rungs 3–7 are all variations on **tiling**: cut the output into blocks, and for each block pull the matching strips of `A` and `B` up the memory pyramid once.

![Tile hierarchy](course/figures/fig_14a_tiling.png)

- The **block tile** lives in **shared memory** — one thread block owns it.
- The **warp tile** is the slice one warp handles.
- The **thread tile** lives in **registers** — one thread accumulates a small `TM×TN` patch.

Each level reuses its loaded data many times before discarding it. A `(128×128)` block tile with `K`-depth `8` loads `128*8 + 8*128 ≈ 2048` values from global memory and uses them in `128*128*8 ≈ 130000` multiply-adds — an intensity roughly 60× the naive kernel. That is how the dot moves right on the roofline.


## 5. Tensor cores and mixed precision

The top rung is special hardware. A **tensor core** is a unit that multiplies two small matrices and accumulates the result in a *single instruction*. Where a normal CUDA core does one multiply-add per cycle, one tensor-core instruction does a whole `16×16×16` matrix multiply (and Blackwell's go far larger). One warp drives one tensor-core operation.

Tensor cores are the reason modern GPUs quote enormous TFLOP numbers — but there's a catch: they want **low-precision inputs**. They read 16-bit values (FP16 or BF16) and **accumulate in 32-bit (FP32)**. So the pattern is:

- Store/feed activations and weights in **BF16** (16 bits — half the memory traffic, and tensor cores eat it natively).
- Let the tensor core **accumulate in FP32** so the running sum doesn't lose precision.

This "16-bit in, 32-bit accumulate" is exactly **mixed precision**, the subject of Chapter 17. For now the takeaway is simple: cuBLAS uses tensor cores when it can, which is why a BF16 matmul on this hardware is several times faster than the same matmul in FP32. BF16's wide exponent range (the same as FP32) is what makes it safe to use for the *inputs* without the loss scaling that FP16 needs.


## 6. cuBLAS vs cuBLASLt

NVIDIA ships two matmul APIs, and `llm.c` uses both flavors' ideas:

- **cuBLAS** — the classic BLAS interface (`cublasSgemm`, `cublasGemmEx`). Simple: hand it pointers, sizes, and transpose flags. Great for a plain `C = A @ B`.
- **cuBLASLt** ("Lt" = *lightweight / tensor*) — a more configurable interface (`cublasLtMatmul`). More setup, but it exposes **epilogues**: extra work fused onto the end of the matmul. This is what `llm.c`'s `matmul_cublaslt` uses.

Both share the one quirk Chapter 14 covered in depth: cuBLAS is **column-major** (FORTRAN heritage), while PyTorch and `llm.c` are **row-major**. The fix is the swap trick — pass `B` first, `A` second, no transposes — and the bytes line up so you get row-major `C = A @ B` back. (Re-read Chapter 14 §2 if that's fuzzy; we won't re-derive it here.)

Both also pick the actual kernel for you. When you call cuBLAS, it consults a **heuristic** that, given your `M, N, K`, data type, and GPU, selects the best pre-tuned kernel from a large internal catalog. That's the "two decades of tuning" you're renting with one function call. In `matmul_cublaslt` this is the `cublasLtMatmulAlgoGetHeuristic` call.


## 7. The fused epilogue — cuBLASLt's superpower

Here is the trick that makes `cublasLt` worth its extra setup. The feed-forward block of a Transformer does three things in a row:

```
y = x @ W1ᵀ      (matmul)
y = y + b1        (add bias)
z = gelu(y)       (activation)
```

Done naively that's **three kernel launches**, and — worse — three round-trips through slow global memory: write `y`, read it back to add bias, write it, read it back to GELU, write it. The matmul itself is compute-bound, but the bias and GELU are pure memory traffic.

cuBLASLt fuses all three into the matmul's **epilogue** — extra work done while the result is still in registers/shared memory, before it's ever written out:

![Fused epilogue: 3 kernels become 1](course/figures/fig_14a_epilogue_fusion.png)

In [`llmc/matmul.cuh`](llmc/matmul.cuh), `matmul_cublaslt` picks the epilogue from a small decision tree:

```cpp
if (has_gelu) {
    // save the pre-GELU values to pre_gelu (the "AUX" buffer) for the backward pass
    epilogue = has_bias ? CUBLASLT_EPILOGUE_GELU_AUX_BIAS   // matmul + bias + GELU
                        : CUBLASLT_EPILOGUE_GELU_AUX;        // matmul + GELU
} else if (has_bias) {
    epilogue = CUBLASLT_EPILOGUE_BIAS;                       // matmul + bias
} else {
    epilogue = CUBLASLT_EPILOGUE_DEFAULT;                    // plain matmul
}
```

Two details worth remembering:

- **`AUX`** means "also write the pre-activation result to a side buffer." The backward pass needs the value that went *into* GELU, so the forward pass saves it. That side buffer is the `fch` (pre-GELU) tensor from Chapter 8's `ActivationTensors`.
- **Backward fuses too.** `CUBLASLT_EPILOGUE_DGELU_BGRADB` does GELU-backward and the bias-gradient *inside* the backward matmul — the same idea, mirrored.

This is why `llm.c`'s FFN is fast despite being only a handful of lines: the expensive part is one fused `cublasLtMatmul` call, not three separate kernels.


## 8. Batched and strided-batched GEMM

So far, one matmul at a time. But attention doesn't want one big matmul — it wants **many small ones, all independent**. This section builds that picture from the ground up.

### Step 1 — Where the many small matmuls come from

Take GPT-2 small. A batch of `B = 8` sequences, each `T = 1024` tokens, split across `H = 12` attention heads of size `d = 64`. Inside attention, each head of each sequence does its *own* score matmul:

```
for each sequence  b in 0..7      (B = 8)
  for each head    h in 0..11     (H = 12)
      scores[b,h] = Q[b,h] @ K[b,h]ᵀ        # (1024×64) @ (64×1024) → (1024×1024)
```

That's `B × H = 8 × 12 = 96` separate matmuls. Crucially they are **independent**: head 3 of sequence 5 shares no data with head 7 of sequence 2. You cannot merge them into one big `A @ B` — the math would mix heads together, which is wrong.

### Step 2 — Why not just loop and call cuBLAS 96 times?

You could. But each of the 96 is small (a `(1024×1024)` output from a thin `K = 64`), so no single one fills the GPU, and every call carries launch + setup overhead. You'd spend much of your time *starting and stopping* matmuls rather than computing them. We want **one launch that does all 96** so the GPU stays saturated.

That single call is a **batched GEMM**.

![Strided-batched GEMM](course/figures/fig_14a_strided_batched.png)

### Step 3 — How one call finds 96 matrices: the *stride*

Here's the only new idea. The 96 `Q` matrices already sit in one contiguous buffer in memory — that's just what the tensor `Q` of shape `[B, H, T, d]` *is*: `96` blocks of `T×d = 1024×64 = 65,536` floats, laid end to end.

So cuBLAS doesn't need 96 pointers. It needs **one base pointer + one number**: how far to jump to get from matrix `i` to matrix `i+1`. That jump is the **stride**.

```
strideQ = T * d = 65,536 floats     # distance from Q[b,h] to the next Q
Q for batch index 0  →  Q + 0 * 65536
Q for batch index 1  →  Q + 1 * 65536
Q for batch index i  →  Q + i * 65536
```

Same for `K` (strideK = 65,536) and for the output scores (strideScores = `T * T` = 1,048,576). One base pointer, one stride each, and cuBLAS walks the whole stack itself.

### Step 4 — A tiny worked example you can hold in your head

Forget attention; shrink it. `batch = 2`, each `A` is `2×3`, each `B` is `3×2`, so each `C` is `2×2`. In memory:

```
A buffer  (2 matrices × 6 floats = 12 floats):
  [ A0:  a a a a a a ][ A1:  a a a a a a ]
    ^offset 0           ^offset 6
  strideA = 6   →  matrix i starts at A + i*6

C buffer  (2 matrices × 4 floats = 8 floats):
  [ C0: c c c c ][ C1: c c c c ]
  strideC = 4   →  result i lands at C + i*4
```

One `cublasSgemmStridedBatched(... batch_count = 2)` call computes `C0 = A0@B0` and `C1 = A1@B1`, places them at offsets `0` and `4`, and never mixes the two. **That is the whole concept** — the §10 demo below runs exactly this (with `batch = 8`) and checks it against a plain CPU loop.

### Step 5 — The API, now that the numbers mean something

```c
cublasSgemmStridedBatched(handle, opA, opB,
                          m, n, k,        // shape of ONE matmul in the batch
                          &alpha,
                          A, lda, strideA,   // base ptr, leading dim, jump between A's
                          B, ldb, strideB,
                          &beta,
                          C, ldc, strideC,   // jump between results
                          batch_count);      // how many (96 for our GPT-2 example)
```

Every argument except the three `stride*` / `batch_count` ones is identical to an ordinary `cublasSgemm`. The strides are the *only* new thing, and you now know they're just "elements from one matrix to the next."

### Where llm.c uses it

`llm.c` reaches the same machinery through `matmul_cublaslt`'s `batch_count` / `strideA` / `strideB` / `strideOut` arguments (search for `CUBLASLT_MATRIX_LAYOUT_BATCH_COUNT` in [`llmc/matmul.cuh`](llmc/matmul.cuh)) — that's the non-flash attention path. And the same idea scaled up to a *group* of **different-shaped** GEMMs in one launch is **grouped GEMM**, which is how Mixture-of-Experts layers route tokens to many experts efficiently.


## 9. See the gap: naive kernel vs cuBLAS

Enough words. Let's measure the naive kernel against `cublasSgemm` and `cublasLtMatmul` on this machine, timed with CUDA events, reported in GFLOP/s.


In [ ]:
!mkdir -p course/ch14a_build


In [ ]:
%%writefile course/ch14a_build/matmul_bench.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <cublasLt.h>

// Naive kernel: one thread per output cell, everything from global memory.
__global__ void matmul_naive(float* C, const float* A, const float* B, int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < M && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < K; k++) acc += A[row*K + k] * B[k*N + col];
        C[row*N + col] = acc;
    }
}

static double gflops(int M, int N, int K, float ms) {
    double flop = 2.0 * (double)M * N * K;
    return flop / (ms * 1e-3) / 1e9;
}

int main(void) {
    int M = 2048, N = 2048, K = 2048;
    size_t szA = (size_t)M*K*4, szB = (size_t)K*N*4, szC = (size_t)M*N*4;

    float *h_A = (float*)malloc(szA), *h_B = (float*)malloc(szB);
    for (size_t i = 0; i < (size_t)M*K; i++) h_A[i] = (float)((i*7) % 13) / 6.0f - 1.0f;
    for (size_t i = 0; i < (size_t)K*N; i++) h_B[i] = (float)((i*11) % 17) / 8.0f - 1.0f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, szA); cudaMalloc(&d_B, szB); cudaMalloc(&d_C, szC);
    cudaMemcpy(d_A, h_A, szA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, szB, cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    float ms;

    // --- 1) Naive kernel ---
    dim3 block(16, 16), grid((N+15)/16, (M+15)/16);
    matmul_naive<<<grid, block>>>(d_C, d_A, d_B, M, N, K);   // warmup
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    matmul_naive<<<grid, block>>>(d_C, d_A, d_B, M, N, K);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("naive kernel    : %7.2f ms   %8.1f GFLOP/s\n", ms, gflops(M,N,K,ms));

    // --- 2) cuBLAS Sgemm (row-major C = A@B via the swap trick) ---
    cublasHandle_t handle; cublasCreate(&handle);
    float alpha = 1.0f, beta = 0.0f;
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K,
                &alpha, d_B, N, d_A, K, &beta, d_C, N);       // warmup
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K,
                &alpha, d_B, N, d_A, K, &beta, d_C, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("cuBLAS Sgemm    : %7.2f ms   %8.1f GFLOP/s\n", ms, gflops(M,N,K,ms));

    // --- 3) cuBLASLt (default epilogue), same swap trick ---
    cublasLtHandle_t lt; cublasLtCreate(&lt);
    cublasLtMatmulDesc_t op;
    cublasLtMatmulDescCreate(&op, CUBLAS_COMPUTE_32F, CUDA_R_32F);
    cublasOperation_t opN = CUBLAS_OP_N;
    cublasLtMatmulDescSetAttribute(op, CUBLASLT_MATMUL_DESC_TRANSA, &opN, sizeof(opN));
    cublasLtMatmulDescSetAttribute(op, CUBLASLT_MATMUL_DESC_TRANSB, &opN, sizeof(opN));
    // Column-major layouts for the swapped product C^T = B^T @ A^T -> first operand B (N x K)
    cublasLtMatrixLayout_t lB, lA, lC;
    cublasLtMatrixLayoutCreate(&lB, CUDA_R_32F, N, K, N);
    cublasLtMatrixLayoutCreate(&lA, CUDA_R_32F, K, M, K);
    cublasLtMatrixLayoutCreate(&lC, CUDA_R_32F, N, M, N);
    size_t ws_size = 32u*1024*1024; void* ws; cudaMalloc(&ws, ws_size);
    cublasLtMatmulPreference_t pref; cublasLtMatmulPreferenceCreate(&pref);
    cublasLtMatmulPreferenceSetAttribute(pref, CUBLASLT_MATMUL_PREF_MAX_WORKSPACE_BYTES,
                                         &ws_size, sizeof(ws_size));
    cublasLtMatmulHeuristicResult_t heur; int found = 0;
    cublasLtMatmulAlgoGetHeuristic(lt, op, lB, lA, lC, lC, pref, 1, &heur, &found);
    if (found) {
        cublasLtMatmul(lt, op, &alpha, d_B, lB, d_A, lA, &beta, d_C, lC, d_C, lC,
                       &heur.algo, ws, ws_size, 0);            // warmup
        cudaDeviceSynchronize();
        cudaEventRecord(start);
        cublasLtMatmul(lt, op, &alpha, d_B, lB, d_A, lA, &beta, d_C, lC, d_C, lC,
                       &heur.algo, ws, ws_size, 0);
        cudaEventRecord(stop); cudaEventSynchronize(stop);
        cudaEventElapsedTime(&ms, start, stop);
        printf("cuBLASLt matmul : %7.2f ms   %8.1f GFLOP/s\n", ms, gflops(M,N,K,ms));
    } else {
        printf("cuBLASLt matmul : no algorithm found\n");
    }

    printf("\n(M=N=K=%d, FP32. cuBLAS is typically ~10x the naive kernel here;\n"
           " the gap widens with BF16 tensor cores.)\n", M);

    cudaFree(ws); cublasLtMatrixLayoutDestroy(lB); cublasLtMatrixLayoutDestroy(lA);
    cublasLtMatrixLayoutDestroy(lC); cublasLtMatmulPreferenceDestroy(pref);
    cublasLtMatmulDescDestroy(op); cublasLtDestroy(lt); cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C); free(h_A); free(h_B);
    return 0;
}


In [ ]:
!nvcc -O2 -lcublas -lcublasLt -o course/ch14a_build/matmul_bench course/ch14a_build/matmul_bench.cu && ./course/ch14a_build/matmul_bench


The naive kernel runs at a small fraction of cuBLAS. Same arithmetic, same hardware — the difference is entirely in how the bytes were moved. That gap *is* the optimization ladder from §4. (Try BF16 inputs and you'd see cuBLAS pull further ahead as tensor cores engage.)


## 10. Batched GEMM in action

A small strided-batched example: multiply a stack of `batch` independent matrices in one call, and check it against a plain CPU loop.


In [ ]:
%%writefile course/ch14a_build/batched_demo.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

int main(void) {
    int batch = 8, M = 32, N = 24, K = 40;
    size_t eA = (size_t)M*K, eB = (size_t)K*N, eC = (size_t)M*N;

    float *h_A = (float*)malloc(batch*eA*4);
    float *h_B = (float*)malloc(batch*eB*4);
    float *h_C = (float*)malloc(batch*eC*4);
    float *h_ref = (float*)malloc(batch*eC*4);
    for (size_t i = 0; i < batch*eA; i++) h_A[i] = (float)(i % 7) / 3.0f - 1.0f;
    for (size_t i = 0; i < batch*eB; i++) h_B[i] = (float)(i % 11) / 5.0f - 1.0f;

    // CPU reference: independent row-major C_b = A_b @ B_b
    for (int b = 0; b < batch; b++)
        for (int m = 0; m < M; m++)
            for (int n = 0; n < N; n++) {
                float s = 0;
                for (int k = 0; k < K; k++) s += h_A[b*eA + m*K + k] * h_B[b*eB + k*N + n];
                h_ref[b*eC + m*N + n] = s;
            }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, batch*eA*4); cudaMalloc(&d_B, batch*eB*4); cudaMalloc(&d_C, batch*eC*4);
    cudaMemcpy(d_A, h_A, batch*eA*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, batch*eB*4, cudaMemcpyHostToDevice);

    cublasHandle_t handle; cublasCreate(&handle);
    float alpha = 1.0f, beta = 0.0f;
    // Row-major swap trick, batched: C^T = B^T @ A^T per batch, stride = one matrix.
    cublasSgemmStridedBatched(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K,
                              &alpha,
                              d_B, N, eB,
                              d_A, K, eA,
                              &beta,
                              d_C, N, eC,
                              batch);
    cudaMemcpy(h_C, d_C, batch*eC*4, cudaMemcpyDeviceToHost);

    float maxerr = 0;
    for (size_t i = 0; i < batch*eC; i++) {
        float e = fabsf(h_C[i] - h_ref[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("strided-batched GEMM  batch=%d  M=%d N=%d K=%d  max diff = %.2e  %s\n",
           batch, M, N, K, maxerr, maxerr < 1e-3 ? "PASS" : "FAIL");

    cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C); free(h_ref);
    return 0;
}


In [ ]:
!nvcc -O2 -lcublas -o course/ch14a_build/batched_demo course/ch14a_build/batched_demo.cu && ./course/ch14a_build/batched_demo


## 11. Translation Bridge

| PyTorch you know | What runs on the GPU |
|---|---|
| `nn.Linear(C, OC)(x)` | `cublasLtMatmul` (matmul + bias fused) |
| `gelu(linear(x))` in an MLP | one `cublasLtMatmul` with `EPILOGUE_GELU_AUX_BIAS` |
| `torch.matmul(A, B)` | `cublasSgemm` / `cublasGemmEx` |
| `torch.bmm(A, B)` (batched) | `cublasSgemmStridedBatched` |
| `F.scaled_dot_product_attention` | batched GEMMs for `Q@Kᵀ` and `scores@V` (+ softmax) |
| autocast / BF16 training | `cublasGemmEx` with BF16 inputs, FP32 accumulate, tensor cores |

The rule from Chapter 14 still holds: **if it's dense linear algebra, call cuBLAS.** This chapter just showed you *what* that call is doing and *why* it wins.


## 12. Common Pitfalls

- **Row-major vs column-major.** The #1 source of wrong answers. Use the swap trick (pass `B` then `A`) and write it once in a wrapper. See Chapter 14 §2.
- **Leading dimension (`lda`/`ldb`/`ldc`) mistakes.** The leading dimension is the stride between columns in column-major land, *not* always the matrix width. For a contiguous `(rows, cols)` column-major matrix it's the number of rows. Get this wrong and you read garbage with no error.
- **Alignment.** `matmul_cublaslt` *requires* 16-byte-aligned pointers (it checks and aborts). Misaligned data silently disables the fast paths or fails outright.
- **Precision surprises.** cuBLAS may use TF32 by default on Ampere+ for FP32 inputs, so "FP32" results can differ from a scalar CPU sum at the `1e-3` level. That's expected, not a bug.
- **Forgetting `-lcublasLt`.** `cublasLt*` symbols live in a separate library from `cublas`. Link both.
- **Tiny matmuls.** For very small `M, N, K` the launch/heuristic overhead dominates; that's exactly where batched/grouped GEMM exists to help.


## 13. TODO Exercise — % of peak

Modify `matmul_bench.cu` to also print, for each method, the **percentage of the GPU's FP32 peak** it achieves. Look up your GPU's FP32 TFLOP/s (for the RTX 4080 SUPER it's roughly **49 TFLOP/s**), and compute `achieved_GFLOPs / (peak_TFLOPs * 1000) * 100`.

Predict before you run: which method gets closest to peak, and why can't even cuBLAS reach 100% on plain FP32?


### Solution

In [ ]:
%%writefile course/ch14a_build/exercise_peak.cu
#include <stdio.h>
#include <cuda_runtime.h>

// Drop-in helper: print GFLOP/s and % of a quoted FP32 peak.
// Add this next to gflops() in matmul_bench.cu and call it after each timing.
static const double PEAK_FP32_TFLOPS = 49.0;  // RTX 4080 SUPER, adjust for your GPU

void report(const char* name, int M, int N, int K, float ms) {
    double g = 2.0 * (double)M * N * K / (ms * 1e-3) / 1e9;
    double pct = g / (PEAK_FP32_TFLOPS * 1000.0) * 100.0;
    printf("%-16s: %8.1f GFLOP/s   (%.1f%% of FP32 peak)\n", name, g, pct);
}

int main(void) {
    // Example with illustrative timings to show the formula; feed real ms in practice.
    report("naive (example)",   2048, 2048, 2048, 120.0f);
    report("cuBLAS (example)",  2048, 2048, 2048, 1.6f);
    printf("\nWhy not 100%%? Plain FP32 doesn't use tensor cores; memory, launch,\n"
           "and tail effects also cost a few percent. BF16 (tensor cores) gets much closer.\n");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch14a_build/exercise_peak course/ch14a_build/exercise_peak.cu && ./course/ch14a_build/exercise_peak


## Further Reading

**Source of truth**

- [cuBLAS Library documentation](https://docs.nvidia.com/cuda/cublas/) — `cublasGemmEx`, `cublasLtMatmul`, `cublasSgemmStridedBatched`, the column-major convention, and the full epilogue list.
- `llmc/matmul.cuh` (`matmul_cublaslt`) in this repo — the production wrapper: epilogue selection, batched/strided arguments, algorithm heuristic.
- `dev/cuda/matmul_forward.cu` in this repo — the naive `kernel1`, the hand-tiled `kernel4`, and the cuBLAS/cuBLASLt versions side by side.

**Going deeper**

- [Matrix Multiplication on NVIDIA's Blackwell (Modular blog, part 1)](https://www.modular.com/blog/matrix-multiplication-on-nvidias-blackwell-part-1-introduction) — the clearest modern intro to GPU GEMM, arithmetic intensity, and the memory hierarchy.
- [Anatomy of a CUDA GEMM: from naive kernels to outperforming cuBLAS on Blackwell](https://medium.com/@emmanuelalo52/anatomy-of-a-cuda-gemm-from-naive-kernels-to-outperforming-cublas-on-blackwell-c394b04b5995) — the full optimization ladder with measured speedups (the source for §4's table).
- [How to Optimize a CUDA Matmul Kernel for cuBLAS-like Performance (Simon Boehm)](https://siboehm.com/articles/22/CUDA-MMM) — the canonical step-by-step SGEMM walkthrough.
- [CUTLASS: Fast Linear Algebra in CUDA C++](https://developer.nvidia.com/blog/cutlass-linear-algebra-cuda/) — NVIDIA's open-source template library showing how these tiles are actually built.
- [New cuBLAS 12.0 Features and Matmul Performance on Hopper](https://developer.nvidia.com/blog/new-cublas-12-0-features-and-matrix-multiplication-performance-on-nvidia-hopper-gpus/) and [Grouped GEMM APIs in cuBLAS](https://developer.nvidia.com/blog/introducing-grouped-gemm-apis-in-cublas-and-more-performance-updates/) — how the library keeps evolving (grouped GEMM = MoE).
- [Pro Tip: cuBLAS Strided Batched Matrix Multiply](https://developer.nvidia.com/blog/pro-tip-cublas-strided-batched-matrix-multiply/) — the batched API used for attention (§8).
- [CUDA C++ Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html) — the "prefer existing libraries" guidance and the arithmetic-intensity reasoning.


## Recap

You can now open the cuBLAS black box and describe what's inside:

- **Matmul dominates** deep-learning runtime (~80%+), so its speed is the model's speed.
- A GEMM is just `M×N` dot products; the **arithmetic is fixed**, so speed is all about **memory movement**.
- The naive kernel is **memory-bound** (≈0.25 FLOP/byte) because it re-reads operands from slow global memory.
- The **optimization ladder** — coalescing → shared-memory tiling → register tiling → vectorized loads → warp tiling → tensor cores — is one long campaign to **reuse each byte higher up the memory pyramid**.
- **Tensor cores** do whole sub-matrix multiplies in one instruction, in **BF16 with FP32 accumulate** (mixed precision).
- **cuBLASLt epilogues** fuse bias + GELU into the matmul; **strided-batched** GEMM powers attention.
- You **measured** the gap: cuBLAS beats a simple kernel by ~10× on the same hardware (more with BF16 tensor cores).

### What's next

**Chapter 15 — Kernel Fusion.** The epilogue trick generalizes. We'll see `fused_residual_forward` (residual + layernorm) and `fused_classifier` (softmax + cross-entropy + first backward step) apply the exact same principle you learned here: **fewer trips to global memory = more speed.**